In [14]:
# Scientific Computing
import pandas as pd
import numpy as np

# Text processing.
import re 
import ftfy
import spacy
import nltk
nltk.download('names')
from nltk import ngrams
from nltk.corpus import names
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


# Initializing tok2vec for lemmatization.
# See "Natural Language Processing in Action" for code reference. 
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler"])

[nltk_data] Downloading package names to
[nltk_data]     C:\Users\clara\AppData\Roaming\nltk_data...
[nltk_data]   Package names is already up-to-date!


In [15]:
Survey_df = pd.read_csv("../Clean_Data_Resources/Survey_Results.csv")

C:\Users\clara\AppData\Local\Temp\ipykernel_47560\3248790261.py:1: DtypeWarning: Columns (0: Course_Code, 1: R_Dropin_Tutoring, 2: R_Online_Active_Participation, 3: R_Online_Engaging_Content, 4: R_Weekly_Emails, 5: T_Online_Experience_Suggestions, 6: T_Other_Suggestions) have mixed types. Specify dtype option on import or set low_memory=False.
  Survey_df = pd.read_csv("../Clean_Data_Resources/Survey_Results.csv")


In [16]:
# Define all survey columns that are made of text.
Text_Cols = [
    "T_Collaboration_Understanding",
    "T_Leader_Performance_Suggestions",
    "T_Other_Suggestions",
    "T_Program_Overall_Suggestions",
]

In [17]:
# Tokenize responses.

# Encode responses that functionally do not give information.
Non_Responses = {"n/a", "na", "none", "nothing", "no", "n"}

# Extract nltk's names.
Name_List = set(n.lower() for n in names.words())

def tokenize_response(text):
    # Guard against NaN and empty strings.
    if not isinstance(text, str) or text.strip() == "":
        return []
    # Guard against non-responses.
    if text.strip().lower().replace("/", "") in Non_Responses:
        return []
    # Guard against single character or numeric-only responses.
    if len(text.strip()) <= 1 or text.strip().isdigit():
        return []
    # Fix broken encodings first, then normalize apostrophes and slashes.
    text = ftfy.fix_text(text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("/", " ")
    doc = nlp(text.lower())
    # Strip punctuation, whitespace, contractions, emails, and names.
    return [tok.text for tok in doc
            if not tok.is_punct
            and not tok.is_space
            and tok.text not in {"n't", "ca", "m", "'m", "ve", "'ve", "re", "'re", "'s", "'d"}
            and not re.match(r'\S+@\S+', tok.text)
            and tok.text not in Name_List]

In [18]:
# Use an apply function to tokenize all columns in Text_Cols:
for col in Text_Cols:
    Survey_df[col + "_tokens"] = Survey_df[col].apply(tokenize_response)

c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger

In [19]:
# Function to enact lemmatization.
def lemmatize_response(tokens):
    if not tokens:
        return []
    doc = nlp(" ".join(tokens))
    return [tok.lemma_ for tok in doc if not tok.is_punct and not tok.is_space]

# Function to enact ngrams, wher n specifies the n...grams. 
def make_ngrams(tokens, n):
    return [" ".join(gram) for gram in ngrams(tokens, n)]

In [20]:
# Lemmatize, then build bigrams and trigrams for each column.
for col in Text_Cols:
    lemma_col = col + "_lemmas"
    Survey_df[lemma_col] = Survey_df[col + "_tokens"].apply(lemmatize_response)

    # Pass n=2 and n=3 to make_ngrams for each lemmatized response.
    Survey_df[col + "_bigrams"]  = Survey_df[lemma_col].apply(lambda x: make_ngrams(x, 2))
    Survey_df[col + "_trigrams"] = Survey_df[lemma_col].apply(lambda x: make_ngrams(x, 3))

c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger

In [21]:
Survey_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34512 entries, 0 to 34511
Data columns (total 63 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Start_Date                                  34512 non-null  str    
 1   End_Date                                    34512 non-null  str    
 2   Duration                                    34512 non-null  int64  
 3   Recorded_Date                               34512 non-null  str    
 4   Response_ID                                 34512 non-null  str    
 5   Source_File                                 34512 non-null  str    
 6   Discipline                                  34512 non-null  str    
 7   Course_Code                                 34512 non-null  object 
 8   Is_Online                                   34512 non-null  bool   
 9   Semester                                    34512 non-null  str    
 10  Year                 

# VADER Sentiment

In [22]:
# Initialize sentiment analyzer.
Vader_Sentimenter = SentimentIntensityAnalyzer()

In [23]:
def vader_score(text, min_tokens=4):
    # Return compound score, skipping responses that are too short to be meaningful.
    if not isinstance(text, str) or text.strip() == "":
        return None
    tokens = text.strip().split()
    if len(tokens) < min_tokens:
        return None
    return Vader_Sentimenter.polarity_scores(text)['compound']

# Run the vader_score funciton with the minimum amount of tokens needed for validity.
for col in Text_Cols:
    Survey_df[col + "_vader"] = Survey_df[col].apply(vader_score)

In [24]:
# Initiate an empty list that will hold vader scores for statements that are on a decent length (4 tokens).
Vader_Summary_Non_Null = []

for col in Text_Cols:
    vader_col = col + "_vader"
    # Only include rows where the original text was a non-null, non-empty string.
    has_text = Survey_df[col].notna() & (Survey_df[col].str.strip() != "")
    scores_all = Survey_df.loc[has_text, vader_col]
    # Drop None values returned by token filter.
    scores = scores_all.dropna()
    unscored = has_text.sum() - len(scores)

    Vader_Summary_Non_Null.append({
        # Strip T_ prefix for cleaner display.
        "Column": col.replace("T_", ""),
        "Responses": has_text.sum(),
        "Scored": len(scores),
        "Unscored Too Short": unscored,
        "Mean": round(scores.mean(), 3),
        "Std": round(scores.std(), 3),
        "Neutral": int((scores == 0.0).sum()),
        "Positive": int((scores > 0).sum()),
        "Negative": int((scores < 0).sum()),
        # Responses with strong positive and negative sentiment above VADER's recommended threshold.
        "Strong Positive (>0.5)": int((scores > 0.5).sum()),
        "Strong Negative (<-0.5)": int((scores < -0.5).sum()),
    })

Vader_Summary_Real_df = pd.DataFrame(Vader_Summary_Non_Null).set_index("Column")
print(Vader_Summary_Real_df.to_string())

                                Responses  Scored  Unscored Too Short   Mean    Std  Neutral  Positive  Negative  Strong Positive (>0.5)  Strong Negative (<-0.5)
Column                                                                                                                                                           
Collaboration_Understanding         30050   28146                1904  0.254  0.371     5567     17789      4790                    7283                      721
Leader_Performance_Suggestions      27568   22309                5259  0.300  0.441     3806     14321      4182                    8949                     1383
Other_Suggestions                    1423     833                 590  0.202  0.404      242       431       160                     223                       32
Program_Overall_Suggestions         24382   18569                5813  0.251  0.382     5080     10727      2762                    5175                      510


In [25]:
# Sample scored responses across sentiment tiers per column.
print()
for col in Text_Cols:
    vader_col = col + "_vader"
    has_text = Survey_df[col].notna() & (Survey_df[col].str.strip() != "")

    # Restrict to scored responses only.
    scored = Survey_df.loc[has_text].dropna(subset=[vader_col])

    # Define tiers by compound score boundaries.
    # VADER compound scores range from -1.0 to +1.0.
    # Boundaries follow VADER's own recommended thresholds, so  positive: > 0.05 and negative: < -0.05.
    tiers = {
        "Strongly Positive": scored[scored[vader_col] > 0.5].nlargest(1, vader_col),
        "Mildly Positive":   scored[(scored[vader_col] > 0) & (scored[vader_col] <= 0.5)].sample(1, random_state=42),
        "Neutral":           scored[scored[vader_col] == 0.0].sample(1, random_state=42) if (scored[vader_col] == 0.0).sum() > 0 else pd.DataFrame(),
        "Mildly Negative":   scored[(scored[vader_col] < 0) & (scored[vader_col] >= -0.5)].sample(1, random_state=42),
        "Strongly Negative": scored[scored[vader_col] < -0.5].nsmallest(1, vader_col),
    }

    # Print out results.
    print(col)
    for tier, df in tiers.items():
        if len(df) == 0:
            continue
        row = df.iloc[0]
        print(f"{tier} Score = {round(row[vader_col], 3)}")
        print(f"{str(row[col])}")
        print()


T_Collaboration_Understanding
Strongly Positive Score = 0.988
Working collaboratively with my SI classmates has significantly deepened my understanding of course concepts by providing opportunities for active engagement and diverse perspectives. Discussing ideas with peers helps clarify my confusion, as hearing different explanations or approaches can make complex material easier to grasp. Collaboration encourages active learning, where I’m not just passively reviewing content but engaging in problem-solving and critical thinking alongside others. Sharing notes, resources, and study strategies also enriches my learning experience, as I can benefit from the group's collective knowledge. Additionally, working together fosters a supportive environment that builds confidence and keeps me accountable to my studies, ensuring I stay on track and prepared for exams. Overall, the collaborative nature of SI has helped me better understand the material, retain vital concepts, and approach proble

In [27]:
# Save results as parquet to preserve the bigram, trigram, and lemma columns.
Survey_df["Course_Code"] = Survey_df["Course_Code"].astype(str)
Survey_df.to_parquet("../Clean_Data_Resources/Survey_df_processed.parquet")